## Camada Silver — `ecommerce_categorias`

Este notebook lê o micro-lote novo da camada **Bronze física** (`squad1/bronze/ecommerce_categorias`), aplica as 10 regras de qualidade (5 técnicas + 5 de negócio), gera logs de DQ e grava a Silver em `append` no caminho físico `squad1/silver/ecommerce_categorias`, **com particionamento físico Hive-style**.

### O que estava errado na versão anterior

A versão anterior:
- **Lia** a Bronze via `spark.table("bronze.ecommerce_categorias_bronze")` — uma tabela gerenciada do Unity Catalog.
- **Gravava** a Silver e os DQ Logs via `df.write.format("delta").mode("append").saveAsTable(...)` — também tabelas gerenciadas do Unity Catalog.

No Databricks Free Edition/Serverless, `saveAsTable(...)` sem um `path` físico explícito grava no storage gerenciado do metastore, e não no caminho `abfss://squad1@.../silver/ecommerce_categorias` que é o necessário.

### O que foi adaptado aqui

1. **Leitura da Bronze**: troquei `spark.table(TABELA_BRONZE)` por uma leitura física via SDK `deltalake`, do mesmo caminho `squad1/bronze/ecommerce_categorias` gravado pelo notebook `feat_squad1_elias_ecommerce_categorias_bronze_implementar_bronze`.
2. **Idempotência**: a verificação de quais `bronze_source_file` já foram processados deixou de consultar `spark.table(TABELA_SILVER)` (Unity Catalog) e passou a consultar a própria tabela Delta física da Silver (`squad1/silver/ecommerce_categorias`), via SDK `deltalake`.
3. **Gravação da Silver**: troquei `saveAsTable` pelo mesmo mecanismo `gravar_delta` (SDK `deltalake`) usado no Bronze corrigido — **generalizei a função para também reconhecer as colunas de partição da Silver** (`silver_processed_year/month/day/hour`), já que no Bronze ela só reconhecia as colunas `bronze_ingest_*`.
4. **Particionamento físico Hive-style da Silver**: adicionei as colunas `silver_processed_year`, `silver_processed_month`, `silver_processed_day` e `silver_processed_hour` (derivadas de `silver_processed_at`) e habilitei `PARTICIONAR_SILVER = True`. Fisicamente, `squad1/silver/ecommerce_categorias` passa a ter subpastas `silver_processed_year=AAAA/silver_processed_month=MM/silver_processed_day=DD/silver_processed_hour=HH/`.
5. **DQ Logs**: também passaram a ser gravados fisicamente em Delta (`squad1/dq_monitoring_logs`, sem particionamento — é uma tabela de controle/auditoria), pelo mesmo mecanismo, eliminando outra tabela gerenciada do Unity Catalog que sofria do mesmo problema.
6. **Sem dependência de notebook externo**: todas as funções auxiliares estão definidas dentro deste próprio notebook.
7. **Regra 7 corrigida** (subcategoria deve ter produto): a referência de produtos deixou de ser lida via Unity Catalog (`bronze.ecommerce_produtos_bronze`) e agora é lida pelo **mesmo mecanismo físico** usado para `ecommerce_categorias` — `squad1/bronze/ecommerce_produtos`, via `ler_delta()`. Se essa tabela física ainda não existir, a Regra 7 trata todas as subcategorias como sem produto (e avisa no log) em vez de falhar o notebook.
8. **Regra 8 corrigida** (`r8_raiz_sem_filho_falhou`): a versão anterior contava as próprias linhas da categoria raiz (`F.count("*").over(Window.partitionBy("id_categoria_str"))`), que nunca é zero — a regra nunca disparava. Agora a contagem é feita corretamente: para cada categoria, conta-se quantas linhas do micro-lote têm `id_categoria_pai_str` apontando para ela (ou seja, quantos filhos diretos ela tem), e a regra falha quando uma categoria raiz (`id_categoria_pai_str` nulo) tem zero filhos diretos no lote.

> **Sobre a primeira execução**: caso a pasta física `squad1/silver/ecommerce_categorias` ainda não exista, todo o conteúdo atual da Bronze física será processado nesta primeira carga da Silver. O mesmo vale para a referência de produtos: se `squad1/bronze/ecommerce_produtos` ainda não tiver sido gravada fisicamente, a Regra 7 será aplicada de forma conservadora (sem produto) até que essa tabela exista.

## 1. Instalação do SDK `deltalake`

In [0]:
%pip install -q deltalake pyarrow

## 2. Imports, parâmetros e credenciais

In [0]:
# Imports e parâmetros

import os
import uuid
import pandas as pd
import pyarrow as pa
from functools import reduce
from datetime import datetime, timezone
from dotenv import load_dotenv

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

from deltalake import DeltaTable
from deltalake.writer import write_deltalake

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# --- Origem física (Bronze, Delta) — gravada pelo notebook Bronze ---
CONTAINER_SQUAD1 = "squad1"
CAMADA_ORIGEM = "bronze"
ENTIDADE_ORIGEM = "ecommerce_categorias"

# --- Destino físico (Silver, Delta) ---
CAMADA_DESTINO = "silver"
ENTIDADE_DESTINO = "ecommerce_categorias"

# --- Destino físico dos logs de DQ (tabela de controle/auditoria) ---
# Gravados na raiz do container squad1: squad1/dq_monitoring_logs
CAMADA_DQ = ""
ENTIDADE_DQ = "dq_monitoring_logs"

# --- Referência de produtos para a Regra 7 ---
# Agora lida pelo MESMO mecanismo físico usado para ecommerce_categorias:
# squad1/bronze/ecommerce_produtos, via SDK deltalake (ler_delta).
CAMADA_PRODUTOS_ORIGEM = "bronze"
ENTIDADE_PRODUTOS_ORIGEM = "ecommerce_produtos"

# --- Particionamento físico Hive-style ---
PARTICIONAR_SILVER = True   # silver_processed_year/month/day/hour
PARTICIONAR_DQ_LOGS = False  # tabela de controle, sem necessidade de partição

RUN_ID = str(uuid.uuid4())

print("Origem física (Bronze):", f"{CONTAINER_SQUAD1}/{CAMADA_ORIGEM}/{ENTIDADE_ORIGEM}")
print("Destino físico (Silver):", f"{CONTAINER_SQUAD1}/{CAMADA_DESTINO}/{ENTIDADE_DESTINO}")
print("Destino físico (DQ Logs):", f"{CONTAINER_SQUAD1}/{ENTIDADE_DQ}")
print("Referência física (Produtos, Regra 7):", f"{CONTAINER_SQUAD1}/{CAMADA_PRODUTOS_ORIGEM}/{ENTIDADE_PRODUTOS_ORIGEM}")
print("RUN_ID:", RUN_ID)

# Definir as credenciais do Service Principal (.env):
load_dotenv("/Workspace/Users/soaress.elias@gmail.com/merca-data-platform-categorias/.env")

CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT_NAME = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

# Opções de storage no formato esperado pelo SDK deltalake (object_store/Azure)
STORAGE_OPTIONS = {
    "account_name": STORAGE_ACCOUNT_NAME,
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "tenant_id": TENANT_ID,
}

# Criar a credencial:
credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

# Criar o DataLakeServiceClient (usado apenas na validação física final):
service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    credential=credential
)
file_system_squad1 = service_client.get_file_system_client(file_system=CONTAINER_SQUAD1)

## 3. Funções auxiliares

`get_delta_path`, `delta_existe` e `gravar_delta`. Além disso, `gravar_delta` foi **generalizada** para reconhecer tanto as colunas de partição do Bronze (`bronze_ingest_*`) quanto as da Silver (`silver_processed_*`). `ler_delta` e `obter_arquivos_ja_processados` são novas — leem a tabela Delta física em vez de uma tabela do Unity Catalog.

In [0]:
def get_delta_path(camada: str, tabela: str, storage_opts: dict) -> str:
    """Monta a URI nativa abfss:// para a tabela Delta, tratando gravação na raiz (camada vazia)."""
    conta = storage_opts.get("account_name")
    if not camada:
        return f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/{tabela}"
    return f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/{camada}/{tabela}"


def delta_existe(camada: str, tabela: str, storage_opts: dict) -> bool:
    """Verifica se já existe uma tabela Delta válida no caminho físico."""
    try:
        DeltaTable(get_delta_path(camada, tabela, storage_opts), storage_options=storage_opts)
        return True
    except Exception:
        return False


def ler_delta(camada: str, tabela: str, storage_opts: dict):
    """Lê uma tabela Delta física via SDK deltalake e converte para DataFrame PySpark."""
    if not delta_existe(camada, tabela, storage_opts):
        raise Exception(f"Tabela Delta física não encontrada: {camada}/{tabela}")
    path = get_delta_path(camada, tabela, storage_opts)
    dt = DeltaTable(path, storage_options=storage_opts)
    pdf = dt.to_pandas()
    return spark.createDataFrame(pdf)


def gravar_delta(df, camada: str, tabela: str, storage_opts: dict, mode: str = "append", particionar: bool = True) -> bool:
    """Grava um DataFrame Spark como Delta físico via SDK deltalake (bypass do Serverless)."""
    path = get_delta_path(camada, tabela, storage_opts)
    modo_real = mode if delta_existe(camada, tabela, storage_opts) else "overwrite"

    try:
        # 1. Conversão para Pandas
        pdf = df.toPandas()

        # 2. Correção OBRIGATÓRIA de fuso horário (evita erro fatal no PyArrow)
        for col_name in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col_name]):
                pdf[col_name] = pdf[col_name].dt.tz_localize(None)

        # 3. Conversão para Tabela Arrow
        tabela_arrow = pa.Table.from_pandas(pdf, preserve_index=False)

        # 4. Definição dinâmica de partições por camada (Bronze e Silver)
        partition_by = None
        if particionar:
            colunas_particao_por_camada = {
                "bronze": ["bronze_ingest_year", "bronze_ingest_month", "bronze_ingest_day", "bronze_ingest_hour"],
                "silver": ["silver_processed_year", "silver_processed_month", "silver_processed_day", "silver_processed_hour"],
            }
            possiveis = colunas_particao_por_camada.get(camada, [])
            colunas_pdf = pdf.columns.tolist()
            partition_by = [c for c in possiveis if c in colunas_pdf] or None

        # 5. Gravação via SDK (bypass do Databricks Serverless)
        write_deltalake(
            table_or_uri=path,
            data=tabela_arrow,
            mode=modo_real,
            storage_options=storage_opts,
            partition_by=partition_by,
            schema_mode="overwrite" if modo_real == "overwrite" else "merge"
        )

        print(f"[Sucesso] Gravado fisicamente em: {path} | Linhas: {len(pdf)}")
        return True

    except Exception as e:
        print(f"[Erro] Falha ao gravar {path}: {str(e)}")
        return False


def obter_arquivos_ja_processados(camada: str, tabela: str, storage_opts: dict, coluna_arquivo: str = "bronze_source_file") -> set:
    """Lê a tabela Delta física (se existir) e retorna o conjunto de arquivos de origem já gravados."""
    if not delta_existe(camada, tabela, storage_opts):
        return set()
    try:
        dt = DeltaTable(get_delta_path(camada, tabela, storage_opts), storage_options=storage_opts)
        pdf = dt.to_pandas(columns=[coluna_arquivo])
        return set(pdf[coluna_arquivo].dropna().unique().tolist())
    except Exception as e:
        print(f"[Aviso] Não foi possível ler arquivos já processados: {e}")
        return set()


def filtrar_micro_lote_novo(df_bronze, arquivos_processados: set, coluna_arquivo: str = "bronze_source_file"):
    """Remove do DataFrame os registros cujo arquivo de origem já foi gravado na Silver física."""
    if not arquivos_processados:
        print("Nenhum registro encontrado na Silver física ainda. Processando lote completo.")
        return df_bronze
    df_processados = spark.createDataFrame([(a,) for a in arquivos_processados], [coluna_arquivo])
    return df_bronze.join(df_processados, on=coluna_arquivo, how="left_anti")

## 4. Ler Bronze física e filtrar o micro-lote novo (idempotência por `bronze_source_file`)

In [0]:
df_bronze = ler_delta(camada=CAMADA_ORIGEM, tabela=ENTIDADE_ORIGEM, storage_opts=STORAGE_OPTIONS)

arquivos_ja_processados = obter_arquivos_ja_processados(
    camada=CAMADA_DESTINO,
    tabela=ENTIDADE_DESTINO,
    storage_opts=STORAGE_OPTIONS,
    coluna_arquivo="bronze_source_file"
)

df_micro_lote = filtrar_micro_lote_novo(
    df_bronze=df_bronze,
    arquivos_processados=arquivos_ja_processados,
    coluna_arquivo="bronze_source_file"
)

qtd = df_micro_lote.count()
print("Registros novos para processar:", qtd)

if qtd == 0:
    dbutils.notebook.exit("Nenhum arquivo novo para processar na Silver.")

display(
    df_micro_lote
    .select("bronze_source_file")
    .dropDuplicates()
    .orderBy("bronze_source_file")
)

## 5. Padronização + Preparação de referências

In [0]:
# Padronização
df_base = (
    df_micro_lote
    .withColumn("id_categoria_str", F.trim(F.col("id_categoria").cast("string")))
    .withColumn("nome_categoria_norm", F.trim(F.col("nome_categoria")))
    .withColumn("id_categoria_pai_str", F.trim(F.col("id_categoria_pai").cast("string")))
)

# Referência para produtos (Regra 7) — lida pelo mesmo mecanismo físico
# via SDK deltalake.
if delta_existe(camada=CAMADA_PRODUTOS_ORIGEM, tabela=ENTIDADE_PRODUTOS_ORIGEM, storage_opts=STORAGE_OPTIONS):
    df_produtos_bronze = ler_delta(
        camada=CAMADA_PRODUTOS_ORIGEM,
        tabela=ENTIDADE_PRODUTOS_ORIGEM,
        storage_opts=STORAGE_OPTIONS
    )
    df_produtos_ref = (
        df_produtos_bronze
        .select(F.col("id_categoria").cast("string").alias("id_categoria_str"))
        .where(F.col("id_categoria_str").isNotNull())
        .groupBy("id_categoria_str")
        .count()
        .withColumnRenamed("count", "qtd_produtos")
    )
    print(f"Referência de produtos carregada de squad1/{CAMADA_PRODUTOS_ORIGEM}/{ENTIDADE_PRODUTOS_ORIGEM}.")
else:
    print(
        f"[Aviso] Tabela Delta física squad1/{CAMADA_PRODUTOS_ORIGEM}/{ENTIDADE_PRODUTOS_ORIGEM} "
        "ainda não encontrada. A Regra 7 tratará todas as subcategorias como sem produto."
    )
    df_produtos_ref = spark.createDataFrame([], "id_categoria_str string, qtd_produtos long")

## 6. Aplicar as 10 regras (Regras 7 e 8 corrigidas)

In [0]:
# Janelas para contagens internas do micro-lote
w_id = Window.partitionBy("id_categoria_str")
w_nome_nivel = Window.partitionBy("id_categoria_pai_str", "nome_categoria_norm")

# Lista de IDs válidos (sem cache)
df_ids_validos = df_base.select("id_categoria_str").distinct()

# Contagem de filhos diretos por categoria — usada na Regra 8.
# Conta, para cada id_categoria_pai_str presente no lote, quantas linhas
# apontam para ele como pai (ou seja, quantos filhos diretos essa categoria tem).
df_contagem_filhos = (
    df_base
    .where(F.col("id_categoria_pai_str").isNotNull())
    .groupBy("id_categoria_pai_str")
    .count()
    .withColumnRenamed("id_categoria_pai_str", "id_categoria_pai_ref")
    .withColumnRenamed("count", "qtd_filhos")
)

df_regras = (
    df_base
    .withColumn("qtd_id_no_lote", F.count("*").over(w_id))
    .withColumn("qtd_nome_no_nivel", F.count("*").over(w_nome_nivel))

    # Join com produtos (Regra 7)
    .join(df_produtos_ref, on="id_categoria_str", how="left")
    .withColumn("qtd_produtos", F.coalesce(F.col("qtd_produtos"), F.lit(0)))

    # ==================== REGRAS TÉCNICAS ====================

    # R1: id_categoria não nulo e único no lote
    .withColumn("r1_id_falhou",
                F.col("id_categoria_str").isNull() |
                (F.col("id_categoria_str") == "") |
                (F.col("qtd_id_no_lote") > 1))

    # R2: nome_categoria obrigatório
    .withColumn("r2_nome_falhou",
                F.col("nome_categoria_norm").isNull() |
                (F.col("nome_categoria_norm") == ""))

    # R3: id_categoria_pai deve existir (usando broadcast join)
    .join(
        broadcast(df_ids_validos.withColumnRenamed("id_categoria_str", "id_pai_valido")),
        F.col("id_categoria_pai_str") == F.col("id_pai_valido"),
        how="left"
    )
    .withColumn("r3_pai_invalido_falhou",
                F.col("id_categoria_pai_str").isNotNull() &
                F.col("id_pai_valido").isNull())

    # R4: Sem ciclos (self-reference)
    .withColumn("r4_ciclo_falhou",
                F.col("id_categoria_str") == F.col("id_categoria_pai_str"))

    # R5: Nome único por nível
    .withColumn("r5_nome_duplicado_nivel_falhou",
                F.col("qtd_nome_no_nivel") > 1)

    # R6: Hierarquia deve ter no máximo 2 níveis
    .withColumn("nivel", F.when(F.col("id_categoria_pai_str").isNull(), 0).otherwise(1))
    .withColumn("r6_hierarquia_invalida_falhou", F.col("nivel") > 1)

    # ==================== REGRAS DE NEGÓCIO ====================

    # R7: Subcategoria deve ter pelo menos 1 produto
    .withColumn("r7_sem_produto_falhou",
                (F.col("id_categoria_pai_str").isNotNull()) &
                (F.col("qtd_produtos") == 0))

    # R8: Categoria raiz deve ter pelo menos 1 subcategoria (corrigido):
    # Antes: contava as próprias linhas da categoria raiz (nunca era zero, a regra
    # nunca disparava). Agora: junta com df_contagem_filhos para saber quantas
    # linhas do lote têm essa categoria como pai (id_categoria_pai_str) e falha
    # quando uma categoria raiz tem zero filhos diretos no lote.
    .join(
        df_contagem_filhos,
        F.col("id_categoria_str") == F.col("id_categoria_pai_ref"),
        how="left"
    )
    .withColumn("qtd_filhos", F.coalesce(F.col("qtd_filhos"), F.lit(0)))
    .withColumn("r8_raiz_sem_filho_falhou",
                (F.col("id_categoria_pai_str").isNull()) &
                (F.col("qtd_filhos") == 0))

    # R9: Sem caracteres perigosos
    .withColumn("r9_caracteres_invalidos_falhou",
                F.col("nome_categoria_norm").rlike(r".*[<> &].*"))

    # R10: Quantidade de categorias raiz (AJUSTE O NÚMERO 12 se necessário)
    .withColumn("total_raiz",
                F.sum(F.when(F.col("id_categoria_pai_str").isNull(), 1).otherwise(0))
                .over(Window.partitionBy(F.lit(1))))
    .withColumn("r10_raiz_invalido_falhou", F.col("total_raiz") != 12)

    # Flag final: linha válida somente se todas as regras passaram
    .withColumn("silver_linha_valida",
                ~(
                    F.col("r1_id_falhou") | F.col("r2_nome_falhou") |
                    F.col("r3_pai_invalido_falhou") | F.col("r4_ciclo_falhou") |
                    F.col("r5_nome_duplicado_nivel_falhou") |
                    F.col("r6_hierarquia_invalida_falhou") |
                    F.col("r7_sem_produto_falhou") | F.col("r8_raiz_sem_filho_falhou") |
                    F.col("r9_caracteres_invalidos_falhou") | F.col("r10_raiz_invalido_falhou")
                ))

    .withColumn("silver_processed_at", F.current_timestamp())

    # Colunas de particionamento físico Hive-style da Silver
    .withColumn("silver_processed_year", F.year(F.col("silver_processed_at")))
    .withColumn("silver_processed_month", F.month(F.col("silver_processed_at")))
    .withColumn("silver_processed_day", F.dayofmonth(F.col("silver_processed_at")))
    .withColumn("silver_processed_hour", F.hour(F.col("silver_processed_at")))

    # Limpeza de colunas auxiliares
    .drop("id_pai_valido", "id_categoria_pai_ref")
)

print("✅ Regras aplicadas com sucesso (versão otimizada para Serverless)")
display(df_regras.select(
    "id_categoria", "nome_categoria", "id_categoria_pai",
    "silver_linha_valida",
    "r1_id_falhou", "r2_nome_falhou", "r3_pai_invalido_falhou",
    "r7_sem_produto_falhou", "qtd_filhos", "r8_raiz_sem_filho_falhou"
).limit(15))

## 7. Resumo das regras

In [0]:
flags = [c for c in df_regras.columns if c.startswith("r") and c.endswith("_falhou")]

display(df_regras.groupBy("silver_linha_valida").count())
display(df_regras.agg(*[F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c) for c in flags]))

## 8. Gerar DQ Logs

In [0]:
regras_config = [
    ("R1 - id_categoria não nulo/único", "r1_id_falhou", "Critica"),
    ("R2 - nome_categoria obrigatório", "r2_nome_falhou", "Critica"),
    ("R3 - id_categoria_pai válido", "r3_pai_invalido_falhou", "Critica"),
    ("R4 - sem ciclos na hierarquia", "r4_ciclo_falhou", "Critica"),
    ("R5 - nome único por nível", "r5_nome_duplicado_nivel_falhou", "Critica"),
    ("R6 - hierarquia 2 níveis", "r6_hierarquia_invalida_falhou", "Aviso"),
    ("R7 - subcategoria com produto", "r7_sem_produto_falhou", "Aviso"),
    ("R8 - raiz com subcategoria", "r8_raiz_sem_filho_falhou", "Aviso"),
    ("R9 - sem caracteres perigosos", "r9_caracteres_invalidos_falhou", "Critica"),
    ("R10 - quantidade de raízes esperada", "r10_raiz_invalido_falhou", "Aviso"),
]

def criar_log_regra(df, nome_regra, coluna_flag, severidade):
    return (
        df.groupBy("bronze_source_file")
        .agg(
            F.count("*").cast("int").alias("qtd_registros_total"),
            F.sum(F.when(F.col(coluna_flag), 1).otherwise(0)).cast("int").alias("qtd_registros_falhos")
        )
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("tabela", F.lit("silver_ecommerce_categorias"))
        .withColumn("regra", F.lit(nome_regra))
        .withColumn("status", F.when(F.col("qtd_registros_falhos") > 0, "FAIL").otherwise("PASS"))
        .withColumn("severidade", F.lit(severidade))
        .withColumn("timestamp_execucao", F.current_timestamp())
        .withColumnRenamed("bronze_source_file", "arquivo_origem")
        .select("run_id", "tabela", "regra", "status", "severidade",
                "qtd_registros_falhos", "qtd_registros_total", "timestamp_execucao", "arquivo_origem")
    )

logs = [criar_log_regra(df_regras, nome, flag, sev) for nome, flag, sev in regras_config]
df_dq_logs = reduce(lambda a, b: a.unionByName(b), logs)

display(df_dq_logs)

## 9. Garantir existência da tabela Delta `dq_monitoring_logs` e gravar logs

In [0]:
# -------------------------------------------------------------------
# Função para garantir que a tabela dq_monitoring_logs exista em Delta
# -------------------------------------------------------------------
def ensure_dq_logs_table():
    path = get_delta_path("", "dq_monitoring_logs", STORAGE_OPTIONS)
    if not delta_existe("", "dq_monitoring_logs", STORAGE_OPTIONS):
        print("Tabela dq_monitoring_logs não encontrada. Criando...")
        # Define o esquema exato da tabela de logs
        schema = StructType([
            StructField("run_id", StringType(), True),
            StructField("tabela", StringType(), True),
            StructField("regra", StringType(), True),
            StructField("status", StringType(), True),
            StructField("severidade", StringType(), True),
            StructField("qtd_registros_falhos", IntegerType(), True),
            StructField("qtd_registros_total", IntegerType(), True),
            StructField("timestamp_execucao", TimestampType(), True),
            StructField("arquivo_origem", StringType(), True),
        ])
        empty_df = spark.createDataFrame([], schema)
        pdf = empty_df.toPandas()
        # Normaliza timezone (evita erro no PyArrow)
        for col in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col]):
                pdf[col] = pdf[col].dt.tz_localize(None)
        table = pa.Table.from_pandas(pdf, preserve_index=False)
        write_deltalake(
            table_or_uri=path,
            data=table,
            mode="overwrite",
            storage_options=STORAGE_OPTIONS,
            partition_by=None,
            schema_mode="overwrite"
        )
        print(f"Tabela dq_monitoring_logs criada em {path}")
    else:
        print("Tabela dq_monitoring_logs já existe.")

# -------------------------------------------------------------------
# Garantir que a tabela Delta exista antes de gravar
# -------------------------------------------------------------------
ensure_dq_logs_table()

# -------------------------------------------------------------------
# Gravar os DQ Logs (append, sem particionamento)
# -------------------------------------------------------------------
sucesso_dq = gravar_delta(
    df=df_dq_logs,
    camada=CAMADA_DQ,
    tabela=ENTIDADE_DQ,
    storage_opts=STORAGE_OPTIONS,
    mode="append",
    particionar=PARTICIONAR_DQ_LOGS
)

if sucesso_dq:
    print("✅ DQ Logs gravados com sucesso em squad1/dq_monitoring_logs (Delta, append)!")
else:
    print("⚠️ Falha ao gravar os DQ Logs. Veja a mensagem de erro acima.")

# -------------------------------------------------------------------
# Gravar a Silver física (particionada) – SEM ALTERAÇÕES
# -------------------------------------------------------------------
sucesso_silver = gravar_delta(
    df=df_regras,
    camada=CAMADA_DESTINO,
    tabela=ENTIDADE_DESTINO,
    storage_opts=STORAGE_OPTIONS,
    mode="append",
    particionar=PARTICIONAR_SILVER
)

if sucesso_silver:
    print("✅ Silver gravada com sucesso (Delta físico particionado via deltalake SDK)!")
else:
    raise Exception("Falha ao gravar a camada Silver. Veja a mensagem de erro acima.")


## 10. Validação final (lógica + física)

In [0]:
print("=" * 80)
print("SILVER CONCLUÍDA - ecommerce_categorias")
print("RUN_ID:", RUN_ID)
print("Registros processados:", df_regras.count())
print("Logs DQ gerados:", df_dq_logs.count())
print("Destino físico Silver:", f"{CONTAINER_SQUAD1}/{CAMADA_DESTINO}/{ENTIDADE_DESTINO}")
print("=" * 80)

display(df_regras.select("bronze_source_file").dropDuplicates().orderBy("bronze_source_file"))
display(df_dq_logs.groupBy("status", "severidade").agg(F.sum("qtd_registros_falhos").alias("falhas")))

# --- Validação física do caminho squad1/silver/ecommerce_categorias ---
print("\nArquivos físicos dentro de squad1/silver/ecommerce_categorias:")
try:
    for item in file_system_squad1.get_paths(path=f"{CAMADA_DESTINO}/{ENTIDADE_DESTINO}", recursive=True):
        tipo = "\U0001F4C1" if item.is_directory else "\U0001F4C4"
        print(f"{tipo} {item.name}")
except Exception as e:
    print("Não foi possível listar o destino Silver:", e)

print("\nValidação Delta + partições físicas (Hive-style):")
if delta_existe(camada=CAMADA_DESTINO, tabela=ENTIDADE_DESTINO, storage_opts=STORAGE_OPTIONS):
    print("Sucesso! Tabela Delta física validada em squad1/silver/ecommerce_categorias.")

    dt = DeltaTable(get_delta_path(CAMADA_DESTINO, ENTIDADE_DESTINO, STORAGE_OPTIONS), storage_options=STORAGE_OPTIONS)
    particoes = dt.partitions()
    if particoes:
        for p in sorted(particoes, key=lambda d: (d.get("silver_processed_year"), d.get("silver_processed_month"), d.get("silver_processed_day"), d.get("silver_processed_hour"))):
            print(
                f"silver_processed_year={p.get('silver_processed_year')}/"
                f"silver_processed_month={p.get('silver_processed_month')}/"
                f"silver_processed_day={p.get('silver_processed_day')}/"
                f"silver_processed_hour={p.get('silver_processed_hour')}"
            )
        print(f"\nTotal de partições físicas: {len(particoes)}")
    else:
        print("Nenhuma partição física encontrada.")
else:
    print("Atenção: a tabela Delta da Silver ainda não foi encontrada no caminho físico esperado.")